# Lab 02 Challenge: worked solution

The challenge asks the agent a question its commands cannot answer, then adds the command it needs. This notebook
runs on its own: it repeats the config cell and the pieces of Parts A and D the challenge uses,
unchanged. At the end are our
answers to Part F's six jobs, with the reasons.

The question:

> When was Helena Holý's latest invoice, and how much was it?

In [1]:
from pathlib import Path
import os
import sqlite3

from dotenv import dotenv_values
from openai import OpenAI, omit

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


## The pieces from the lab

The read-only database and the two lookups from Part A, unchanged.

In [2]:
db = sqlite3.connect(f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro", uri=True, check_same_thread=False)


def lookup_invoices(name):
    # How many invoices the customer with this full name has, and their total
    return db.execute(
        "SELECT COUNT(i.InvoiceId), ROUND(SUM(i.Total), 2) "
        "FROM customers c JOIN invoices i ON i.CustomerId = c.CustomerId "
        "WHERE c.FirstName || ' ' || c.LastName = ?",
        (name,),
    ).fetchone()


def lookup_rep(name):
    # The full name of the customer's support rep, or None when no customer has this name
    row = db.execute(
        "SELECT e.FirstName || ' ' || e.LastName "
        "FROM customers c JOIN employees e ON e.EmployeeId = c.SupportRepId "
        "WHERE c.FirstName || ' ' || c.LastName = ?",
        (name,),
    ).fetchone()
    return row[0] if row else None

The protocol from Part D: the system prompt with three commands and `run_command`.

In [3]:
client = OpenAI()

AGENT_SYSTEM = """You are the support assistant for Chinook, an online music shop. You cannot see the shop's database yourself.
Reply with exactly one line, and make it one of these commands:
LOOKUP_INVOICES <customer full name>  (returns how many invoices the customer has and their total)
LOOKUP_REP <customer full name>  (returns the customer's support rep)
DONE <your answer to the user>
Look up every fact before you answer. Never answer from memory."""


def run_command(line):
    # Run one LOOKUP_ line and return the result as text for the model
    command, _, name = line.partition(" ")
    if command == "LOOKUP_INVOICES":
        count, total = lookup_invoices(name)
        return f"{count} invoices, total {total}" if count else f"no customer named {name}"
    if command == "LOOKUP_REP":
        return lookup_rep(name) or f"no customer named {name}"
    return "unknown command: reply with LOOKUP_INVOICES, LOOKUP_REP or DONE"

And the loop with its turn cap, from Step 7.

In [4]:
def run_agent(question, max_turns=5):
    # Let the model pick lookups until it replies DONE or the turn cap is reached
    messages = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": question},
    ]
    usages = []
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, max_completion_tokens=100, reasoning_effort=EFFORT or omit
        )
        usages.append(response.usage)
        line = response.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": line})
        if line.startswith("DONE"):
            break
        messages.append({"role": "user", "content": "Result: " + run_command(line)})
    return messages, usages


LATEST = "When was Helena Holý's latest invoice, and how much was it?"

## Task 1: Run the agent as it is

No command returns a date. What does the model do with the commands it has?

In [5]:
messages, usages = run_agent(LATEST)
for message in messages[1:]:
    print(f"{message['role']:9} {message['content']}")
print("model calls:", len(usages))

user      When was Helena Holý's latest invoice, and how much was it?
assistant LOOKUP_INVOICES Helena Holý
user      Result: 7 invoices, total 49.62
assistant DONE Helena Holý has 7 invoices totaling 49.62; I can’t determine the date or amount of her latest invoice from that result.
model calls: 2


No command fits, so the model used the closest one, `LOOKUP_INVOICES`, read the count and total, and said in its `DONE` line that it cannot tell the date or amount of the latest invoice from that result. It did not invent a date. Two model calls.

One run shows one choice. Five more runs, with the commands and the `DONE` line of each:

In [6]:
for run in range(5):
    messages, usages = run_agent(LATEST)
    steps = [m["content"].split()[0] for m in messages if m["role"] == "assistant"]
    print(f"run {run + 1}: {' -> '.join(steps)}")
    print("       ", messages[-1]["content"])

run 1: LOOKUP_INVOICES -> DONE
        DONE Helena Holý has 7 invoices totaling 49.62, but I can’t determine which was the latest or its amount from that result.


run 2: LOOKUP_INVOICES -> DONE
        DONE Helena Holý has 7 invoices totaling 49.62.


run 3: LOOKUP_INVOICES -> DONE
        DONE I can’t determine when Helena Holý’s latest invoice was from the invoice count and total alone.


run 4: LOOKUP_INVOICES -> DONE
        DONE Helena Holý has 7 invoices totaling $49.62, but I can’t determine the date or amount of her latest invoice from that result.


run 5: LOOKUP_INVOICES -> DONE
        DONE I can see Helena Holý has 7 invoices totaling 49.62, but I don’t have the date or amount of her latest invoice.


The same two commands in all five runs. Four `DONE` lines say the date is not in the result. Run 2 answers with the count and total only, which is not what was asked, and does not say so. No run invents a date. With no command for the latest invoice, the best the agent can do is say that it cannot tell, and in one run of five it did not even do that.

## Task 2: Add a command for the latest invoice

First the answer we expect, from our own query: Helena is customer 6, and the newest invoice is the first row when we
sort by date, newest first.

In [7]:
db.execute("SELECT InvoiceId, InvoiceDate, Total FROM invoices WHERE CustomerId = 6 ORDER BY InvoiceDate DESC LIMIT 1").fetchone()

(404, '2013-11-13 00:00:00', 25.86)

Invoice 404, on 2013-11-13, for 25.86. Now the lookup by name, the same shape as `lookup_invoices`: `ORDER BY
i.InvoiceDate DESC` puts the newest invoice first and `LIMIT 1` keeps only that row. `substr(i.InvoiceDate, 1, 10)`
keeps the first ten characters, the date, and drops the time, which is always `00:00:00` in this database.

In [8]:
def lookup_latest_invoice(name):
    # The date and total of the customer's newest invoice, or None when no customer has this name
    return db.execute(
        "SELECT substr(i.InvoiceDate, 1, 10), i.Total "
        "FROM customers c JOIN invoices i ON i.CustomerId = c.CustomerId "
        "WHERE c.FirstName || ' ' || c.LastName = ? "
        "ORDER BY i.InvoiceDate DESC LIMIT 1",
        (name,),
    ).fetchone()


lookup_latest_invoice("Helena Holý")

('2013-11-13', 25.86)

The same date and total as our query, so the lookup is right.

Now the protocol: one new line in the system prompt, saying what the command returns, and one new branch in
`run_command`. Nothing else changes. `run_agent` reads `AGENT_SYSTEM` and `run_command` by name each time it runs, so
redefining them here is enough.

In [9]:
AGENT_SYSTEM = """You are the support assistant for Chinook, an online music shop. You cannot see the shop's database yourself.
Reply with exactly one line, and make it one of these commands:
LOOKUP_INVOICES <customer full name>  (returns how many invoices the customer has and their total)
LOOKUP_REP <customer full name>  (returns the customer's support rep)
LOOKUP_LATEST_INVOICE <customer full name>  (returns the date and total of the customer's latest invoice)
DONE <your answer to the user>
Look up every fact before you answer. Never answer from memory."""


def run_command(line):
    # Run one LOOKUP_ line and return the result as text for the model
    command, _, name = line.partition(" ")
    if command == "LOOKUP_INVOICES":
        count, total = lookup_invoices(name)
        return f"{count} invoices, total {total}" if count else f"no customer named {name}"
    if command == "LOOKUP_REP":
        return lookup_rep(name) or f"no customer named {name}"
    if command == "LOOKUP_LATEST_INVOICE":
        row = lookup_latest_invoice(name)
        return f"latest invoice on {row[0]}, total {row[1]}" if row else f"no customer named {name}"
    return "unknown command: reply with LOOKUP_INVOICES, LOOKUP_REP, LOOKUP_LATEST_INVOICE or DONE"


print(run_command("LOOKUP_LATEST_INVOICE Helena Holý"))

latest invoice on 2013-11-13, total 25.86


The same question, the same `run_agent`:

In [10]:
messages, usages = run_agent(LATEST)
for message in messages[1:]:
    print(f"{message['role']:9} {message['content']}")
print("model calls:", len(usages))

user      When was Helena Holý's latest invoice, and how much was it?
assistant LOOKUP_LATEST_INVOICE Helena Holý
user      Result: latest invoice on 2013-11-13, total 25.86
assistant DONE Helena Holý's latest invoice was on 2013-11-13, and it totaled $25.86.
model calls: 2


The model picked the new command on its first call, read the result and answered with 2013-11-13 and $25.86, the values of our query. Two model calls: the trace the challenge expected.

## Task 3: Check the new agent over five runs

The check looks for the two values as the database writes them, the date `2013-11-13` and the total `25.86`, in the
`DONE` line of each run.

In [11]:
passed = 0
for run in range(5):
    messages, usages = run_agent(LATEST)
    steps = [m["content"].split()[0] for m in messages if m["role"] == "assistant"]
    final = messages[-1]["content"]
    ok = final.startswith("DONE") and "2013-11-13" in final and "25.86" in final
    passed += ok
    print(f"run {run + 1}: {' -> '.join(steps):30} date and total: {'yes' if ok else 'no'}")

print(f"\n{'✅' if passed == 5 else '❌'} date and total in {passed} of 5 runs")

run 1: LOOKUP_LATEST_INVOICE -> DONE  date and total: yes


run 2: LOOKUP_LATEST_INVOICE -> DONE  date and total: yes


run 3: LOOKUP_LATEST_INVOICE -> DONE  date and total: yes


run 4: LOOKUP_LATEST_INVOICE -> DONE  date and total: yes


run 5: LOOKUP_LATEST_INVOICE -> DONE  date and total: yes

✅ date and total in 5 of 5 runs


`LOOKUP_LATEST_INVOICE -> DONE` in all 5 runs, with the date and the total in every `DONE` line.

What changed between task 1 and task 2 is the list of options, not a list of steps. We never told the model to use
`LOOKUP_LATEST_INVOICE` for this question; it read the command's description in the system prompt and chose it. That
is the agent's side of "who decides the next step". The workflow from Part B would need a new step written by us.

## Part F: our answers to the six jobs

Rows 4 and 5 have more than one answer you could defend; these are ours, with the reason for each.

| # | Job | Shape | Who decides the next step? | Autonomy |
|---|---|---|---|---|
| 1 | Send the same overdue reminder to every account more than 30 days late | workflow | our code: the list and the message are known | act with audit |
| 2 | A visitor asks what jazz fusion is | chatbot | the model writes a reply; there is no next step | suggest |
| 3 | A messy email with an invoice question, an address change and a complaint | agent | the model: the next lookup depends on what it finds | draft |
| 4 | Refund duplicate charges straight from the inbox, up to a set limit, with no one in the loop | agent-shaped, but the wrong default | the model would choose, and a wrong refund costs money | draft, or a rules workflow |
| 5 | Route each incoming request to billing, shipping or product | either | a three-way label is a workflow; an agent only for the requests that fit no label | act with audit on the clear ones, draft for the rest |
| 6 | Find out why revenue fell last quarter | agent | the model: the next query depends on the last result | suggest |

- **Row 1.** If you can write the steps down in advance, it is a workflow. The reminders are logged and can be
  resent or withdrawn, so `act with audit`. An agent here spends model calls to rediscover an order we already know.
- **Row 2.** Talking is the whole job: no lookup, no loop.
- **Rows 3 and 6.** Nobody can write the path before seeing the email or the numbers, which is the case for an
  agent. Row 6 stays at `suggest`, because a wrong cause turns into a wrong business decision.
- **Row 4.** It looks like an agent's job, but it must not run at `act`: a wrong refund is paid before anyone sees
  it. A small rules workflow ("refund the later of two same-day invoices for the same album, up to the limit") or an
  agent at `draft` is the safer answer.
- **Row 5.** If the three categories are stable, a classifier and three queues is a workflow you can test. If many
  requests need two lookups before anyone knows the team, those are the agent's share.